# ============================================================
# PRUEBA TECNICA - FASTFOOD ANALYTICS
# Notebook: 04_silver_transformacion
# Descripcion: Limpieza, transformacion y cruce de datos
#              Tiendas + Sensores usando distancia Haversine
# Autor: Rafael Milanes
# Fecha: 2025-03
# ============================================================


In [1]:
# Bronze
BRONZE_TIENDAS  = "Files/bronze/mysql/tiendas"
BRONZE_VENTAS   = "Files/bronze/mysql/ventas"
BRONZE_TICKET   = "Files/bronze/mysql/ticket"
BRONZE_PRODUCT  = "Files/bronze/mysql/product"
BRONZE_TYPE     = "Files/bronze/mysql/type"
BRONZE_REGION   = "Files/bronze/mysql/region"
BRONZE_SENSORES = "Files/bronze/mongodb/ubicacion_sensores"
BRONZE_EVENTOS  = "Files/bronze/mongodb/sensor_eventos"

# Silver
SILVER_TIENDAS  = "Files/silver/tiendas_clean"
SILVER_VENTAS   = "Files/silver/ventas_clean"
SILVER_SENSORES = "Files/silver/sensores_clean"
SILVER_CRUZADO  = "Files/silver/ventas_sensores_cruzado"

print("Rutas cargadas correctamente")

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 3, Finished, Available, Finished, False)

Rutas cargadas correctamente


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

# Leer todas las tablas Bronze
df_tiendas  = spark.read.format("delta").load(BRONZE_TIENDAS)
df_ventas   = spark.read.format("delta").load(BRONZE_VENTAS)
df_ticket   = spark.read.format("delta").load(BRONZE_TICKET)
df_product  = spark.read.format("delta").load(BRONZE_PRODUCT)
df_type     = spark.read.format("delta").load(BRONZE_TYPE)
df_region   = spark.read.format("delta").load(BRONZE_REGION)
df_sensores = spark.read.format("delta").load(BRONZE_SENSORES)
df_eventos  = spark.read.format("delta").load(BRONZE_EVENTOS)

print("Tablas cargadas desde Bronze:")
print(f"  Tiendas:  {df_tiendas.count()} filas")
print(f"  Ventas:   {df_ventas.count()} filas")
print(f"  Ticket:   {df_ticket.count()} filas")
print(f"  Sensores: {df_sensores.count()} filas")
print(f"  Eventos:  {df_eventos.count()} filas")

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 4, Finished, Available, Finished, False)

Tablas cargadas desde Bronze:
  Tiendas:  10 filas
  Ventas:   85936 filas
  Ticket:   208863 filas
  Sensores: 20 filas
  Eventos:  175200 filas


In [3]:
print("=== TIENDAS ===")
df_tiendas.printSchema()
df_tiendas.show(3)

print("=== SENSORES ===")
df_sensores.printSchema()
df_sensores.show(5)

print("=== EVENTOS ===")
df_eventos.printSchema()
df_eventos.show(3)

print("=== VENTAS ===")
df_ventas.printSchema()
df_ventas.show(3)

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 5, Finished, Available, Finished, False)

=== TIENDAS ===
root
 |-- id: long (nullable = true)
 |-- region_id: long (nullable = true)
 |-- tamano_id: long (nullable = true)
 |-- empleados: long (nullable = true)
 |-- ubicacion: string (nullable = true)

+---+---------+---------+---------+--------------------+
| id|region_id|tamano_id|empleados|           ubicacion|
+---+---------+---------+---------+--------------------+
|  4|        1|        2|       10|4.724168, -74.071331|
|  5|        1|        2|       12|4.689609, -74.051762|
|  3|        4|        3|       16|4.585923, -74.089871|
+---+---------+---------+---------+--------------------+
only showing top 3 rows

=== SENSORES ===
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- region_id: long (nullable = true)

+---+----------+----------------+---------+
| id|      name|       ubicacion|region_id|
+---+----------+----------------+---------+
|  2|sensor_002|  4.69, -74.0723|        1|
|  4|sensor_004|4

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Parsear coordenadas de tiendas desde string "lat, lon"
df_tiendas_clean = df_tiendas \
    .withColumn("lat", F.split(F.col("ubicacion"), ",")[0].cast(DoubleType())) \
    .withColumn("lon", F.split(F.col("ubicacion"), ",")[1].cast(DoubleType())) \
    .dropna(subset=["lat", "lon"])

df_tiendas_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(SILVER_TIENDAS)

print(f"OK Tiendas Silver: {df_tiendas_clean.count()} filas")
df_tiendas_clean.show()

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 6, Finished, Available, Finished, False)

OK Tiendas Silver: 10 filas
+---+---------+---------+---------+--------------------+--------+----------+
| id|region_id|tamano_id|empleados|           ubicacion|     lat|       lon|
+---+---------+---------+---------+--------------------+--------+----------+
|  4|        1|        2|       10|4.724168, -74.071331|4.724168|-74.071331|
|  5|        1|        2|       12|4.689609, -74.051762|4.689609|-74.051762|
|  9|        4|        1|        7|4.557695, -74.213006|4.557695|-74.213006|
| 10|        4|        1|        9|4.555034, -74.140505|4.555034|-74.140505|
|  3|        4|        3|       16|4.585923, -74.089871|4.585923|-74.089871|
|  7|        2|        2|       14|4.627556, -74.139330|4.627556| -74.13933|
|  6|        3|        2|       15|4.616622, -74.072496|4.616622|-74.072496|
|  2|        2|        3|       17|4.674784, -74.129521|4.674784|-74.129521|
|  1|        1|        3|       20|4.716154, -74.038500|4.716154|  -74.0385|
|  8|        3|        1|        5|4.576474, -74

In [5]:
# Parsear coordenadas de sensores
df_sensores_clean = df_sensores \
    .withColumn("lat", F.split(F.col("ubicacion"), ",")[0].cast(DoubleType())) \
    .withColumn("lon", F.split(F.col("ubicacion"), ",")[1].cast(DoubleType())) \
    .dropna(subset=["lat", "lon"])

df_sensores_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(SILVER_SENSORES)

print(f"OK Sensores Silver: {df_sensores_clean.count()} filas")
df_sensores_clean.show()

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 7, Finished, Available, Finished, False)

OK Sensores Silver: 20 filas
+---+----------+----------------+---------+------+--------+
| id|      name|       ubicacion|region_id|   lat|     lon|
+---+----------+----------------+---------+------+--------+
|  7|sensor_007|4.6498, -74.1018|        2|4.6498|-74.1018|
| 12|sensor_012|4.5913, -74.0932|        3|4.5913|-74.0932|
|  8|sensor_008|4.6303, -74.1321|        2|4.6303|-74.1321|
| 13|sensor_013| 4.582, -74.0845|        3| 4.582|-74.0845|
| 15|sensor_015|4.5687, -74.2022|        4|4.5687|-74.2022|
| 16|sensor_016| 4.5512, -74.212|        4|4.5512| -74.212|
|  6|sensor_006| 4.6502, -74.122|        2|4.6502| -74.122|
|  9|sensor_009|4.6299, -74.1419|        2|4.6299|-74.1419|
| 10|sensor_010| 4.6104, -74.102|        3|4.6104| -74.102|
|  2|sensor_002|  4.69, -74.0723|        1|  4.69|-74.0723|
|  4|sensor_004|4.6706, -74.0128|        1|4.6706|-74.0128|
|  1|sensor_001|4.7099, -74.0522|        1|4.7099|-74.0522|
| 14|sensor_014|4.5726, -74.0714|        3|4.5726|-74.0714|
| 17|sensor

In [6]:
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

df_eventos_clean = df_eventos \
    .withColumn("id",        F.col("id").cast(IntegerType())) \
    .withColumn("Sensor_id", F.col("Sensor_id").cast(IntegerType())) \
    .withColumn("valor",     F.col("valor").cast(DoubleType())) \
    .withColumn("fecha",     F.to_timestamp(F.col("fecha"), "yyyy-MM-dd HH:mm:ss")) \
    .dropna()

print(f"OK Eventos clean: {df_eventos_clean.count()} filas")
df_eventos_clean.show(3)

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 8, Finished, Available, Finished, False)

OK Eventos clean: 175200 filas
+-----+---------+------------------+-------------------+
|   id|Sensor_id|             valor|              fecha|
+-----+---------+------------------+-------------------+
|21583|        3|1021.8067341798877|2024-06-18 06:00:00|
|21592|        3| 1034.906546872479|2024-06-18 15:00:00|
|21593|        3|1017.6350388554548|2024-06-18 16:00:00|
+-----+---------+------------------+-------------------+
only showing top 3 rows



In [7]:
import math
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# Funcion distancia Haversine en km
def haversine(lat1, lon1, lat2, lon2):
    try:
        R = 6371
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * \
            math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    except:
        return None

haversine_udf = udf(haversine, DoubleType())

# Cruzar todas las tiendas con todos los sensores
df_cruce = df_tiendas_clean.alias("t") \
    .crossJoin(df_sensores_clean.alias("s")) \
    .withColumn("distancia_km", haversine_udf(
        F.col("t.lat"), F.col("t.lon"),
        F.col("s.lat"), F.col("s.lon")
    ))

# Quedarse con el sensor mas cercano por tienda
from pyspark.sql.window import Window

window = Window.partitionBy("t.id").orderBy("distancia_km")
df_sensor_cercano = df_cruce \
    .withColumn("rank", F.rank().over(window)) \
    .filter(F.col("rank") == 1) \
    .select(
        F.col("t.id").alias("tienda_id"),
        F.col("t.ubicacion").alias("tienda_ubicacion"),
        F.col("t.lat").alias("tienda_lat"),
        F.col("t.lon").alias("tienda_lon"),
        F.col("s.id").alias("sensor_id"),
        F.col("s.name").alias("sensor_nombre"),
        F.col("distancia_km")
    )

print(f"OK Cruce tienda-sensor: {df_sensor_cercano.count()} filas")
df_sensor_cercano.show()

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 9, Finished, Available, Finished, False)

OK Cruce tienda-sensor: 10 filas
+---------+--------------------+----------+----------+---------+-------------+------------------+
|tienda_id|    tienda_ubicacion|tienda_lat|tienda_lon|sensor_id|sensor_nombre|      distancia_km|
+---------+--------------------+----------+----------+---------+-------------+------------------+
|        1|4.716154, -74.038500|  4.716154|  -74.0385|        1|   sensor_001|1.6699071829353012|
|        2|4.674784, -74.129521|  4.674784|-74.129521|        5|   sensor_005|2.0543330929445593|
|        3|4.585923, -74.089871|  4.585923|-74.089871|       12|   sensor_012|0.7025851438940883|
|        4|4.724168, -74.071331|  4.724168|-74.071331|        1|   sensor_001|2.6479710528289466|
|        5|4.689609, -74.051762|  4.689609|-74.051762|        3|   sensor_003|1.5796551873138311|
|        6|4.616622, -74.072496|  4.616622|-74.072496|       11|   sensor_011| 1.352261099308597|
|        7|4.627556, -74.139330|  4.627556| -74.13933|        9|   sensor_009|0.38609

In [11]:
# Precipitacion promedio por sensor por dia
df_precip_diaria = df_eventos_clean \
    .withColumn("fecha_dia", F.to_date(F.col("fecha"))) \
    .groupBy("Sensor_id", "fecha_dia") \
    .agg(
        F.avg("valor").alias("precip_promedio"),
        F.max("valor").alias("precip_max"),
        F.min("valor").alias("precip_min")
    ).withColumnRenamed("Sensor_id", "sensor_id_precip") \
     .withColumnRenamed("fecha_dia", "fecha_dia_precip")

# Ventas + ticket + tienda + sensor + precipitacion
df_ventas_sensor = df_ventas \
    .join(df_ticket, df_ventas.factura_id == df_ticket.factura_id, "left") \
    .withColumn("fecha_dia", F.to_date(F.col("fecha_venta"))) \
    .join(df_sensor_cercano,
        df_ventas.tienda_id == df_sensor_cercano.tienda_id, "left") \
    .join(df_precip_diaria,
        (F.col("sensor_id") == F.col("sensor_id_precip")) &
        (F.col("fecha_dia") == F.col("fecha_dia_precip")), "left") \
    .select(
        df_ventas.id.alias("venta_id"),
        df_ventas.tienda_id,
        df_ventas.factura_id,
        F.col("fecha_venta"),
        F.col("fecha_dia"),
        F.col("product_id"),
        F.col("tipo_compra_id"),
        F.col("sensor_id"),
        F.col("sensor_nombre"),
        F.col("distancia_km"),
        F.col("precip_promedio"),
        F.col("precip_max"),
        F.col("precip_min")
    )

df_ventas_sensor.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(SILVER_CRUZADO)

print(f"OK Silver cruzado: {df_ventas_sensor.count()} filas")
df_ventas_sensor.show(3)

StatementMeta(, 286e2819-04ae-4241-95eb-d973d812b4af, 13, Finished, Available, Finished, False)

OK Silver cruzado: 208863 filas
+--------+---------+----------+-------------------+----------+----------+--------------+---------+-------------+------------------+------------------+-----------------+------------------+
|venta_id|tienda_id|factura_id|        fecha_venta| fecha_dia|product_id|tipo_compra_id|sensor_id|sensor_nombre|      distancia_km|   precip_promedio|       precip_max|        precip_min|
+--------+---------+----------+-------------------+----------+----------+--------------+---------+-------------+------------------+------------------+-----------------+------------------+
|   21485|        5|     21485|2024-03-31 19:00:00|2024-03-31|        15|             2|        3|   sensor_003|1.5796551873138311|1078.4686198158124|1193.133647754547|1007.0572710534657|
|   21485|        5|     21485|2024-03-31 19:00:00|2024-03-31|        11|             2|        3|   sensor_003|1.5796551873138311|1078.4686198158124|1193.133647754547|1007.0572710534657|
|   21486|        6|     214